In [2]:
import pandas as pd
import numpy as np
from pathlib import Path
import matplotlib.pyplot as plt
import matplotlib
import re

In [3]:
funda_csv_path = Path('../data/nfq/nfqcsv')
funda_files = [f for f in funda_csv_path.rglob('.') if f.suffix == '.csv']
len(funda_files)

4608

In [6]:
subject_counter = {}
tmp_csv = pd.read_csv(funda_files[0], nrows=5)
for col in tmp_csv.columns:
    subject_counter[col] = []

In [8]:
for csv in funda_files:
    df = pd.read_csv(csv)
    count = df.count()
    len_df = len(df["Ticker"])
    for iter_count, subject in zip(count, df.columns.to_list()):
        subject_counter[subject].append(iter_count/len_df)

columns = list(subject_counter.keys())
data = subject_counter
df = pd.DataFrame(columns=columns, data=data)
df.to_csv("../data/nfq/funda_features_research/funda_appearance_rate.csv") # data\nfq\funda_features_research

In [9]:
arrange_rate = 0.0

In [10]:
df = pd.read_csv("../data/nfq/funda_features_research/funda_appearance_rate.csv")
matplotlib.rcParams["font.family"] = "Meiryo"  # メイリオ

input_Length = len(df.columns)

delete_columns = {"delete":[]}

for column in df.columns.to_list():
    rate_lst = df[column].to_list()
    rate_lst.sort()
    # ある項目の更新頻度が，全ての銘柄において全くない場合
    if rate_lst[int(len(rate_lst))-1] <= arrange_rate: 
        delete_columns["delete"].append(column)
        df.drop(columns=column, inplace=True)

output_Length = len(df.columns)

print(f"Remove elements with a 0% update frequency: {input_Length} -> {output_Length}")

# 列数
ncols = int(len(df.columns.to_list()) ** 0.5)
nrows = int(np.ceil(len(df.columns) / ncols))

# サブプロット作成
fig, axes = plt.subplots(nrows, ncols, figsize=(ncols*2, nrows*2.5))
axes = axes.flatten()

bins = np.arange(0, 1.01, 0.05)

# 各列ごとにヒストグラム
for i, col in enumerate(df.columns):
    df[col].dropna().hist(ax=axes[i], bins=bins, edgecolor="black")
    axes[i].set_xlim(-0.5,1.5)
    axes[i].set_ylim(-0.5,4000)
    axes[i].set_title(re.sub(r"([^）］].)(／|［)", r"\1\n\2", col), fontsize=8, wrap=True)

# 余った軸を消す
for j in range(i+1, len(axes)):
    fig.delaxes(axes[j])

plt.tight_layout()
plt.savefig(f"../data/nfq/funda_features_research/deleted_{input_Length}-{output_Length}.png", dpi=200) # ../data/nfq/funda_features_research/funda_appearance_rate.csv
plt.close()

output_df = pd.DataFrame(data=delete_columns, columns=["delete"]).to_csv("../data/nfq/funda_features_research/delete_funda_list.csv")

Remove elements with a 0% update frequency: 463 -> 449
